In [2]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [3]:
pip install -U langchain-community sentence-transformers

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_5480/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/jayasri.csv")
documents = loader.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0
)

documents = text_splitter.split_documents(documents)

In [7]:
from langchain_community.vectorstores import Chroma

In [8]:
pip install chromadb

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/factory.py", line 169, in _make_candidate_from_dist
    base = self._installed_candidate_cache[dist.canonical_name]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
KeyError: 'huggingface-hub'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/d

In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents,
    embeddings
)

In [13]:

# create retriever
retriever = vectorstore.as_retriever()

In [14]:
pip install langchain-groq groq

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq


# =========================
# 2. LLM (GROQ)
# =========================
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",   # ✅ NEW working model
    temperature=0
)


In [16]:
!pip install langchain langchain-community langchain-groq --upgrade

In [17]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [18]:

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)


# =========================
# TEST QUERY
# =========================
query = "is jayasri marreied"

compressed_docs = compression_retriever.invoke(query)

compressed_docs

[Document(metadata={'row': 0, 'source': '/content/jayasri.csv'}, page_content='"Category: Personal Information,Full Name,Jayasri T\nField: None\nDetails": None'),
 Document(metadata={'source': '/content/jayasri.csv', 'row': 47}, page_content='"Category: Job Application Details,Applied To,Microsoft Pvt. Ltd. – Chennai 600 001\nField: None\nDetails": None'),
 Document(metadata={'source': '/content/jayasri.csv', 'row': 7}, page_content='"Category: Contact Information,Email,jayasri21072006@gmail.com\nField: None\nDetails": None'),
 Document(metadata={'row': 10, 'source': '/content/jayasri.csv'}, page_content='"Category: Education,Institution,Anand Institute of Higher Technology (AIHT) - Chennai\nField: None\nDetails": None')]

In [19]:
!pip install langchain langchain-core langchain-community -q

In [20]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser

# Prompt template (fixed extra quote in original)
template = """You are a helpful assistant that answers questions based on the following context.
If you don't find the answer in the context, just say that you don't know.

Context: {context}

Question: {input}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Helper to format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG pipeline
rag_chain = (
    {
        "context": compression_retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Invoke
response = rag_chain.invoke("what jayasri knowe")
print(response)

Jayasri T knows Tamil and English.
